# CFD vs AI Benchmark
Compare OpenFOAM results with PINN predictions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath('../../'))

from core.pinn_engine import PINN
from core.data_pipeline import CFDDataset

# Load data
dataset = CFDDataset('data/cylinder_flow')
try:
    x, u = dataset.load_simulation('case1')
except FileNotFoundError as e:
    print(f"Warning: {e}. Please ensure data is available.")
    # Create dummy data for demonstration
    x = np.random.rand(100, 2).astype(np.float32)
    u = np.sin(x[:, 0:1]) * np.cos(x[:, 1:2])

# Load pre-trained model
model = PINN()
if os.path.exists('models/cylinder_pinn.pt'):
    model.load_state_dict(torch.load('models/cylinder_pinn.pt'))
else:
    print("Warning: Pre-trained model not found. Using randomly initialized model.")

# Predict
x_tensor = torch.tensor(x[:,0:1], dtype=torch.float32)
t_tensor = torch.zeros_like(x_tensor)  # steady state
u_pred = model.predict(x_tensor, t_tensor).numpy()

# Error
error = np.mean((u - u_pred)**2)
print(f'MSE: {error:.6f}')

# Plot
plt.scatter(x[:,0], x[:,1], c=u[:,0], alpha=0.6)
plt.title('CFD velocity (Dummy/Real)')
plt.colorbar()
plt.show()